In [1]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from io import StringIO
import os
import smtplib
from email.message import EmailMessage
import yfinance as yf
import time

# 1. Configuration & Data Loading

In [2]:
file_path = r'C:\Users\hhaid\OneDrive\Desktop\Cornell Tech stuff\Github\REER Project\Trading Partners.xlsx'
# Load the Excel file
df = pd.read_excel(file_path)
df.head() # Check if file loaded properly

,Country,ISO3,Currency,Currency Code,Weight (2018),CPI,CPI*,P*,FX_Rate_Base(2010),FX_Rate_Current,Weight_Decimal,RPI_i,Partner_FX_Ratio,Nominal_Index_i,REER_Component
0,China,CHN,Chinese Yuan Renminbi,CNY,32.1476,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,United States,USA,United States Dollar,USD,10.0547,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Germany,DEU,Euro,EUR,6.6308,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Japan,JPN,Japanese Yen,JPY,4.8880,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,India,IND,Indian Rupee,INR,3.3896,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 2. Fetching Latest CPI Data (Base year 2010=100)

In [5]:
import requests
import time

def get_wb_cpi_bulk(iso3_list):
    """Fetch most recent CPI (2010=100) for multiple countries in a single request."""
    codes = ';'.join(iso3_list)
    url = f"https://api.worldbank.org/v2/country/{codes}/indicator/FP.CPI.TOTL"
    params = {'format': 'json', 'mrnev': 1, 'gapfill': 'Y', 'per_page': 1000}
    headers = {'User-Agent': 'Mozilla/5.0'}

    r = requests.get(url, params=params, headers=headers, timeout=20)

    if r.status_code != 200:
        print(f"Bulk request failed: status={r.status_code}")
        print(r.text[:500])  # show the actual error message from the API
        return {}

    data = r.json()
    if len(data) < 2 or not data[1]:
        print("Bulk request returned no data")
        return {}

    results = {}
    for entry in data[1]:
        iso3 = entry['countryiso3code']
        value = entry['value']
        date = entry['date']
        if value is not None:
            results[iso3] = (float(value), date)
    return results


# Build full list including Pakistan
all_iso3 = df['ISO3'].dropna().unique().tolist() + ['PAK']
cpi_results = get_wb_cpi_bulk(all_iso3)

df['CPI'] = df['ISO3'].map(lambda c: cpi_results.get(c, (None, None))[0])
df['CPI_Year'] = df['ISO3'].map(lambda c: cpi_results.get(c, (None, None))[1])

pak_cpi, pak_cpi_year = cpi_results.get('PAK', (None, None))
print(f"Pakistan's CPI (2010=100) = {pak_cpi} (year: {pak_cpi_year})")

missing_cpi = df[df['CPI'].isna()][['Country', 'ISO3']]
if not missing_cpi.empty:
    print("Missing World Bank CPI data for:")
    print(missing_cpi)

df.head()  # Check the updated DataFrame with CPI data

Pakistan's CPI (2010=100) = 400.518188961503 (year: 2025)
Missing World Bank CPI data for:
   Country ISO3
16  Taiwan  TWN


,Country,ISO3,Currency,Currency Code,Weight (2018),CPI,CPI*,P*,FX_Rate_Base(2010),FX_Rate_Current,Weight_Decimal,RPI_i,Partner_FX_Ratio,Nominal_Index_i,REER_Component,CPI_Year
0,China,CHN,Chinese Yuan Renminbi,CNY,32.1476,132.596516,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
1,United States,USA,United States Dollar,USD,10.0547,143.857336,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024
2,Germany,DEU,Euro,EUR,6.6308,137.797659,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
3,Japan,JPN,Japanese Yen,JPY,4.8880,118.043593,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
4,India,IND,Indian Rupee,INR,3.3896,233.063138,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025


# 3. Fetching FX Rates: Base Historical (2010) & Current

In [6]:
from curl_cffi import requests as curl_requests
import warnings
warnings.filterwarnings('ignore', category=FutureWarning, module='yfinance')
session = curl_requests.Session(impersonate="chrome")

#Get FX rates for all currencies using Yahoo Finance
def get_fx_rates(currency_code, session):
    if currency_code == 'USD':
        return 1.0, 1.0
    ticker_symbol = f"{currency_code}=X"
    try:
        ticker = yf.Ticker(ticker_symbol, session=session)
        hist_2010 = ticker.history(start='2010-01-01', end='2010-01-15')
        rate_2010 = hist_2010['Close'].iloc[0] if not hist_2010.empty else None
        hist_current = ticker.history(period='5d')
        rate_current = hist_current['Close'].iloc[-1] if not hist_current.empty else None
        return rate_2010, rate_current
    except Exception as e:
        print(f"Failed to fetch {ticker_symbol}: {e}")
        return None, None

# Fetch FX rates for all unique currency codes to avoid redundancy
fx_cache = {}
for code in df['Currency Code'].dropna().unique():
    fx_cache[code] = get_fx_rates(code, session)
    time.sleep(1)

# Map FX rates to the DataFrame
df['FX_Rate_Base(2010)'] = df['Currency Code'].map(lambda c: fx_cache.get(c, (None, None))[0])
df['FX_Rate_Current'] = df['Currency Code'].map(lambda c: fx_cache.get(c, (None, None))[1])

failed = df[df['FX_Rate_Base(2010)'].isna() | df['FX_Rate_Current'].isna()][['Country', 'Currency Code']]
if not failed.empty:
    print("Missing FX data for:")
    print(failed)

# Fetch Pakistan's FX
pak_fx_base, pak_fx_current = get_fx_rates('PKR', session)

if pak_fx_base is None or pak_fx_current is None:
    raise ValueError("Failed to fetch Pakistan's FX rates — cannot proceed with REER calc")

print(f"Pakistan's FX Rate (2010 Base) = {pak_fx_base:.2f}, Current FX Rate = {pak_fx_current:.2f}")
df.head()

Pakistan's FX Rate (2010 Base) = 83.34, Current FX Rate = 277.34


,Country,ISO3,Currency,Currency Code,Weight (2018),CPI,CPI*,P*,FX_Rate_Base(2010),FX_Rate_Current,Weight_Decimal,RPI_i,Partner_FX_Ratio,Nominal_Index_i,REER_Component,CPI_Year
0,China,CHN,Chinese Yuan Renminbi,CNY,32.1476,132.596516,NaN,NaN,6.816900,6.761200,NaN,NaN,NaN,NaN,NaN,2025
1,United States,USA,United States Dollar,USD,10.0547,143.857336,NaN,NaN,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN,2024
2,Germany,DEU,Euro,EUR,6.6308,137.797659,NaN,NaN,0.694930,0.876100,NaN,NaN,NaN,NaN,NaN,2025
3,Japan,JPN,Japanese Yen,JPY,4.8880,118.043593,NaN,NaN,92.919998,162.994003,NaN,NaN,NaN,NaN,NaN,2025
4,India,IND,Indian Rupee,INR,3.3896,233.063138,NaN,NaN,46.610001,96.529999,NaN,NaN,NaN,NaN,NaN,2025


# 4. Calculating REER 

In [7]:
# Clean weights (ensure they sum to 1.0, e.g., 15% = 0.15)
df['Weight_Decimal'] = df['Weight (2018)'] / 100

# 1. Calculate Relative Price Index (RPI) for each country
df['RPI_i'] = pak_cpi / df['CPI']

# 2. Calculate Nominal Exchange Rate Index for each country
# Formula: (Partner Current FX / Partner Base FX) / (Pak Current FX / Pak Base FX)
df['Partner_FX_Ratio'] = df['FX_Rate_Current'] / df['FX_Rate_Base(2010)']
pak_fx_ratio = pak_fx_current / pak_fx_base

df['Nominal_Index_i'] = df['Partner_FX_Ratio'] / pak_fx_ratio

# 3. Apply the Weights to the product of Nominal Index and RPI
df['REER_Component'] = np.power((df['Nominal_Index_i'] * df['RPI_i']), df['Weight_Decimal'])

# Safety net: check for and handle any countries with missing components
missing = df[df['REER_Component'].isna()][['Country', 'CPI', 'FX_Rate_Base(2010)', 'FX_Rate_Current']]
if not missing.empty:
    print("⚠️ Excluding these countries from REER due to missing data:")
    print(missing)

    # Renormalize weights to only the countries with complete data
    valid = df.dropna(subset=['REER_Component'])
    renorm_weight = valid['Weight_Decimal'] / valid['Weight_Decimal'].sum()
    df.loc[valid.index, 'REER_Component'] = np.power(
        (valid['Nominal_Index_i'] * valid['RPI_i']), renorm_weight
    )


# 4. Final REER is the product of all components, scaled by 100
REER_Final = df['REER_Component'].prod() * 100
print(f"Final REER = {REER_Final:.2f}")

# Save the final calculated dataset locally
output_file = 'REER_Calculated_Results.csv'
df.to_csv(output_file, index=False)

⚠️ Excluding these countries from REER due to missing data:
   Country  CPI  FX_Rate_Base(2010)  FX_Rate_Current
16  Taiwan  NaN               31.66           32.375
Final REER = 102.49


# 5. Sending as Email

In [ ]:
def send_reer_email(reer_value, attachment_path):
    """Sends the REER results via email using environment variables for security."""
    
    
    sender_email = os.environ.get('EMAIL_USER')
    sender_password = os.environ.get('EMAIL_PASS') # Use an App Password if using Gmail
    receiver_email = "recipient@example.com" # Replace with your target email
    
    if not sender_email or not sender_password:
        print("Email credentials not found in environment variables. Email skipped.")
        return

    msg = EmailMessage()
    msg['Subject'] = 'Automated REER Calculation Results'
    msg['From'] = sender_email
    msg['To'] = receiver_email
    
    # Email Body
    msg.set_content(f"""
    Hello,
    
    The automated Real Effective Exchange Rate (REER) calculation has completed successfully.
    
    Current Pakistan FX (USD/PKR): {pak_fx_current:.2f}
    Calculated REER: {reer_value:.2f}
    
    The detailed breakdown per country is attached to this email.
    """)

    # Attach the CSV
    try:
        with open(attachment_path, 'rb') as f:
            file_data = f.read()
            file_name = f.name
        
        msg.add_attachment(file_data, maintype='text', subtype='csv', filename=file_name)
        
        # Send via Gmail SMTP server (Adjust if using Outlook/Yahoo)
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
            smtp.login(sender_email, sender_password)
            smtp.send_message(msg)
            print("Email sent successfully!")
            
    except Exception as e:
        print(f"Failed to send email: {e}")

# Trigger the email function
send_reer_email(REER_Final, output_file)